# 07 — Implementatie: studentprognose draaien via PyPi

Deze notebook is een **blauwdruk voor instellingen** die de tool willen overnemen op
hun eigen infrastructuur — een cloud-notebook, een Azure ML workspace, een interne
JupyterHub, of een laptop zonder repo-clone.

> 🔗 **Aanvullend bij:** [`docs/aan-de-slag.md` — Cloud-gebruik](../docs/aan-de-slag.md#cloud-gebruik-data-al-in-memory).

## Wat is anders dan 00–06?

| | Notebooks 00–06 (uitleg) | Notebook 07 (implementatie) |
|---|---|---|
| Repo-clone vereist | Ja | **Nee** |
| Importeert | Helpers + private modules | Alleen publieke API (`__all__`) |
| Demodata | `data/input/*.csv` lokaal | GitHub raw URL |
| Doel | De methodologie *begrijpen* | De tool *draaien* op eigen data |

Volg de stappen hieronder; vervang aan het eind de demo-URL door je eigen databron.


## 1. Installatie

In een schone omgeving (vers virtual env, Google Colab, Azure ML, …):

```bash
pip install studentprognose
```

> 💡 Deze notebook hoeft niet uit deze repo te komen — download `07_implementatie.ipynb`
> van GitHub en open in je eigen Jupyter-omgeving.


## 2. Demodata ophalen via de GitHub raw API

Voor demonstratie lezen we de Radboud-demodata rechtstreeks van GitHub. Dit illustreert
de pattern: laad DataFrames in-memory en geef ze door aan de pipeline. Vervang de URL's
door je eigen ETL-output (lokaal pad, Azure Blob, S3, database-query).

Voor het cumulatieve spoor zijn twee bestanden nodig:
- `vooraanmeldingen_cumulatief.csv` — wekelijkse stand van vooraanmeldingen
- `student_count_first-years.xlsx` — historische realisatie (eerstejaars per opleiding × jaar)


In [1]:
import pandas as pd

RAW = "https://raw.githubusercontent.com/cedanl/studentprognose/main/data/input"

df_cum = pd.read_csv(f"{RAW}/vooraanmeldingen_cumulatief.csv", sep=";", skiprows=[1])
df_sc = pd.read_excel(f"{RAW}/student_count_first-years.xlsx")

print(f"Cumulatief:           {len(df_cum):,} regels, {df_cum.shape[1]} kolommen")
print(f"Historische realisatie: {len(df_sc):,} regels")
df_cum.head(3)

Cumulatief:           25,271 regels, 16 kolommen
Historische realisatie: 432 regels


,Korte naam instelling,Collegejaar,Weeknummer rapportage,Weeknummer,Faculteit,Type hoger onderwijs,Groepeernaam Croho,Naam Croho opleiding Nederlands,Croho,Herinschrijving,Hogerejaars,Herkomst,Gewogen vooraanmelders,Ongewogen vooraanmelders,Aantal aanmelders met 1 aanmelding,Inschrijvingen
0,21PB,2016,1,1,SOW,Bachelor,B Psychologie,B Psychologie,56604,Nee,Nee,EER,2.33,3,NaN,NaN
1,21PB,2016,1,1,SOW,Bachelor,B Psychologie,B Psychologie,56604,Nee,Nee,Niet-EER,1.55,2,NaN,NaN
2,21PB,2016,1,1,MED,Bachelor,B Geneeskunde,B Geneeskunde,56551,Nee,Nee,NL,22.48,29,NaN,NaN


## 3. De pipeline draaien

`run_pipeline_from_dataframes` is de publieke entry-point uit `studentprognose.__all__`.
Hij accepteert DataFrames in-memory, slaat de ETL-stap over, en geeft het resultaat
terug als DataFrame. `save_output=False` voorkomt dat er bestanden op disk worden geschreven —
handig in cloud-omgevingen zonder schrijfrechten.


In [2]:
import os
import tempfile
from studentprognose import run_pipeline_from_dataframes, DataOption

PREDICT_YEAR = 2023
PREDICT_WEEK = 12

# Tijdelijke werkmap zodat de pipeline geen bestanden in je huidige map schrijft
with tempfile.TemporaryDirectory() as tmp:
    os.makedirs(os.path.join(tmp, "data", "output"), exist_ok=True)
    result = run_pipeline_from_dataframes(
        year=PREDICT_YEAR,
        week=PREDICT_WEEK,
        data_cumulative=df_cum,
        data_student_numbers=df_sc,
        dataset=DataOption.CUMULATIVE,
        cwd=tmp,
        save_output=False,
    )

print(f"Resultaat: {len(result):,} rijen × {result.shape[1]} kolommen")
result[
    [
        "Croho groepeernaam",
        "Herkomst",
        "Examentype",
        "SARIMA_cumulative",
        "Prognose_ratio",
        "Baseline",
    ]
].head(10)

Preprocessing...

  Dataset:       cumulatief
  Trainingsdata: jaren 2016-2024
  Voorspelling:  jaar [2023], vanaf week [12]
  Voorspelt:     week 13 t/m 38 (26 weken vooruit)
  Opleidingen:   B Bedrijfskunde, B Biomedische Wetenschappen, B Communicatiewetenschap, B Filosofie, B Geneeskunde, B Geschiedenis, B Kunstmatige Intelligentie, B Natuurkunde, B Politicologie, B Psychologie, B Scheikunde, B Wiskunde, M Bestuurskunde, M Cognitive Neuroscience, M Informatica, M Linguistiek, M Molecular Mechanisms of Disease, M Rechtsgeleerdheid

Predicting first-years: 2023-12...
Start parallel predicting...


Postprocessing...
Resultaat: 1,458 rijen × 23 kolommen


,Croho groepeernaam,Herkomst,Examentype,SARIMA_cumulative,Prognose_ratio,Baseline
0,B Bedrijfskunde,EER,Bachelor,65.0,61.834636,61.834636
1,B Bedrijfskunde,NL,Bachelor,265.0,299.346780,299.346780
2,B Bedrijfskunde,Niet-EER,Bachelor,22.0,23.371719,23.371719
3,B Biomedische Wetenschappen,EER,Bachelor,22.0,23.228768,23.228768
4,B Biomedische Wetenschappen,NL,Bachelor,206.0,186.443486,186.443486
5,B Biomedische Wetenschappen,Niet-EER,Bachelor,9.0,8.470588,8.470588
6,B Communicatiewetenschap,EER,Bachelor,20.0,18.720000,18.720000
7,B Communicatiewetenschap,NL,Bachelor,200.0,218.871696,218.871696
8,B Communicatiewetenschap,Niet-EER,Bachelor,6.0,4.864865,4.864865
9,B Filosofie,EER,Bachelor,12.0,9.880366,9.880366


## 4. Resultaat verwerken

Het resultaat is een gewone pandas DataFrame. Schrijf naar je eigen bestemming:


In [3]:
# Optie A — naar Parquet voor data-lake
# result.to_parquet("voorspellingen_2023_wk12.parquet", index=False)

# Optie B — naar Excel voor handmatige analyse
# result.to_excel("voorspellingen_2023_wk12.xlsx", index=False)

# Optie C — terug naar cloud (voorbeeld voor Azure Blob)
# from azure.storage.blob import BlobServiceClient
# blob.upload_blob(result.to_csv(index=False), overwrite=True)

# Voor demo: laat de top-5 SARIMA-voorspellingen zien
top5 = result.nlargest(5, "SARIMA_cumulative")
top5[["Croho groepeernaam", "Herkomst", "SARIMA_cumulative", "Prognose_ratio"]]

,Croho groepeernaam,Herkomst,SARIMA_cumulative,Prognose_ratio
28,B Psychologie,NL,485.0,496.505639
13,B Geneeskunde,NL,381.0,428.491484
19,B Kunstmatige Intelligentie,NL,289.0,315.991662
1,B Bedrijfskunde,NL,265.0,299.346780
4,B Biomedische Wetenschappen,NL,206.0,186.443486


## 5. Op je eigen data

Vervang stap 2 door je eigen databron. Het schema moet matchen met wat de pipeline
verwacht (zie [`docs/je-data-voorbereiden.md`](../docs/je-data-voorbereiden.md) voor
kolomnamen). Een typische instellings-implementatie ziet er zo uit:

```python
import pandas as pd
from studentprognose import run_pipeline_from_dataframes, DataOption

# 1. Laad jouw cumulatieve telbestand (Studielink-export, intern datawarehouse, …)
df_cum = pd.read_csv("/pad/naar/eigen_aanbod.csv", sep=";", skiprows=[1])

# 2. Laad je historische realisatie eerstejaars
df_sc = pd.read_excel("/pad/naar/eigen_realisatie.xlsx")

# 3. Optioneel: laad ook de individuele aanmelddata voor het volledige ensemble
# df_ind = pd.read_csv("/pad/naar/individueel.csv", sep=";", skiprows=[1])

# 4. Draai de pipeline
result = run_pipeline_from_dataframes(
    year=2025,
    week=12,
    data_cumulative=df_cum,
    data_student_numbers=df_sc,
    # data_individual=df_ind,        # voor -d both
    # dataset=DataOption.BOTH_DATASETS,
    dataset=DataOption.CUMULATIVE,
    save_output=False,
)
```

## Wat als ik meer wil?

- **Eigen configuratie** (numerus-fixus opleidingen aanpassen, modelgewichten, …):
  geef een `configuration=…` dict mee aan `run_pipeline_from_dataframes`. Zie de tweede
  code-block onder [`Cloud-gebruik`](../docs/aan-de-slag.md#cloud-gebruik-data-al-in-memory).
- **Methodologie begrijpen**: open `00_overzicht.ipynb` t/m `06_output_interpreteren.ipynb`
  vanuit een repo-clone — die geven inzicht in wat er onder de motorkap gebeurt.
- **CLI-gebruik** in plaats van API: `studentprognose -d c -y 2025 -w 12` na `pip install`.
